# Notebook 2 — Curation, Deduplication & Descriptors
**Dataset Creation, Curation, and Analysis of GPCR Receptor Targets**

Takes `data/raw_filtered.csv` from Notebook 1 (already MW < 700 and Phase I-IV filtered) and
finishes the pipeline exactly as described in the thesis:

1. **Salt removal** -> parent structure (RDKit `SaltRemover`; largest-fragment fallback)
2. **Peptide removal** -> drop molecules with >= 2 amide-backbone bonds
3. **pKi** = ChEMBL `pchembl_value`
4. **Deduplicate** on (molecule, target), keeping the highest pKi
5. **15 molecular descriptors** (+ aromatic proportion, parent SMILES)

Output: `data/curated_15desc.csv` -> the locked dataset (expected **3,011 records / 719 molecules /
47 targets / 18 families**). Each step logs its surviving counts so the drop table is complete.

In [ ]:
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Crippen, rdMolDescriptors, QED
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

OUTDIR = 'data'
IN_CSV  = os.path.join(OUTDIR, 'raw_filtered.csv')   # from Notebook 1
OUT_CSV = os.path.join(OUTDIR, 'curated_15desc.csv')
print('RDKit ready.')

In [ ]:
# --- Load Notebook 1 output ---
df = pd.read_csv(IN_CSV)
print(f'loaded {len(df)} rows | {df.molecule_chembl_id.nunique()} molecules | '
      f'{df.target.nunique()} targets')

funnel = {'0. from Notebook 1 (MW+phase filtered)': (len(df), df.molecule_chembl_id.nunique())}
def step(name, d):
    funnel[name] = (len(d), d['molecule_chembl_id'].nunique()); return d

In [ ]:
# --- 1. Salt removal -> parent structure ---
sr = SaltRemover()
def to_parent(smiles):
    m = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
    if m is None:
        return None
    p = sr.StripMol(m, dontRemoveEverything=True)
    if p is None or p.GetNumAtoms() == 0:
        frags = Chem.GetMolFrags(m, asMols=True)
        p = max(frags, key=lambda x: x.GetNumAtoms()) if frags else m
    return p

df['mol'] = df['canonical_smiles'].apply(to_parent)
df = step('1. parsable + salt-stripped', df[df['mol'].notna()].copy())
df['parent_smiles'] = df['mol'].apply(Chem.MolToSmiles)

In [ ]:
# --- 2. Peptide / biologic removal ---
# Primary filter: ChEMBL's authoritative `molecule_type` (fetched in Notebook 1).
# Small molecules are retained; peptides, proteins, oligosaccharides, etc. are dropped.
# This is what produced the locked 3,011-record dataset. A generic 'two amide groups'
# rule is WRONG here: many ordinary drugs contain two amides and would be lost.
if 'molecule_type' in df.columns:
    keep = df['molecule_type'].fillna('Small molecule').str.contains('Small molecule', case=False)
    df = step('2. small molecules only (peptides/biologics removed)', df[keep].copy())
else:
    # Fallback if molecule_type is unavailable: genuine peptide BACKBONE only
    # (>=2 consecutive peptide bonds N-Ca-C(=O)-N-Ca-C(=O)), not any amide.
    pep = Chem.MolFromSmarts('[NX3][CX4][CX3](=O)[NX3][CX4][CX3](=O)')
    df = step('2. peptides removed (backbone rule)', df[~df['mol'].apply(lambda m: m.HasSubstructMatch(pep))].copy())

In [ ]:
# --- 3. pKi + 4. deduplicate (highest pKi per molecule-target pair) ---
df['pKi'] = pd.to_numeric(df['pchembl_value'], errors='coerce')
df = df[df['pKi'].notna()]
df = df.sort_values('pKi', ascending=False).drop_duplicates(['molecule_chembl_id', 'target'])
df = step('3. deduplicated (highest pKi / pair)', df.copy())

In [ ]:
# --- 5. Fifteen molecular descriptors (+ aromatic proportion) ---
def aromatic_proportion(m):
    heavy = [a for a in m.GetAtoms() if a.GetAtomicNum() > 1]
    return sum(a.GetIsAromatic() for a in heavy) / len(heavy) if heavy else 0.0

def halogen_count(m):
    return sum(a.GetSymbol() in ('F', 'Cl', 'Br', 'I') for a in m.GetAtoms())

def descriptors(m, pKi):
    MW   = Descriptors.MolWt(m)
    LogP = Crippen.MolLogP(m)
    RotB = Lipinski.NumRotatableBonds(m)
    AP   = aromatic_proportion(m)
    # ESOL aqueous solubility (Delaney 2004)
    LogS = 0.16 - 0.63 * LogP - 0.0062 * MW + 0.066 * RotB - 0.74 * AP
    HAC  = m.GetNumHeavyAtoms()
    LE   = 1.37 * pKi / HAC if HAC else np.nan          # ~ dG per heavy atom
    return {
        'MW': MW, 'LogP': LogP, 'LogS': LogS,
        'HBA': Lipinski.NumHAcceptors(m), 'HBD': Lipinski.NumHDonors(m),
        'TPSA': rdMolDescriptors.CalcTPSA(m), 'RotB': RotB,
        'RingCount': rdMolDescriptors.CalcNumRings(m), 'HAC': HAC,
        'Fsp3': rdMolDescriptors.CalcFractionCSP3(m), 'HalogenCount': halogen_count(m),
        'QED': QED.qed(m),
        'LE': LE, 'LLE': pKi - LogP, 'LELP': (LogP / LE) if LE else np.nan,
        'AromaticProportion': AP,
    }

desc = df.apply(lambda r: descriptors(r['mol'], r['pKi']), axis=1, result_type='expand')
df = pd.concat([df.reset_index(drop=True), desc.reset_index(drop=True)], axis=1)
print('descriptors computed for', len(df), 'records')

In [ ]:
# --- Assemble & save (exact column order of the locked dataset) ---
COLS = ['molecule_chembl_id','target','family','uniprot','max_phase','pKi','MW','LogP','LogS',
        'HBA','HBD','TPSA','RotB','RingCount','HAC','Fsp3','HalogenCount','QED','LE','LLE','LELP',
        'AromaticProportion','parent_smiles']
final = df[COLS].copy()
final.to_csv(OUT_CSV, index=False)

print('CURATION FUNNEL  (records | unique molecules)')
print('-'*54)
for name,(nr,nm) in funnel.items():
    print(f'  {nr:6d} | {nm:5d}   {name}')
print('-'*54)
print(f'FINAL    : {len(final):5d} records | {final.molecule_chembl_id.nunique()} molecules | '
      f'{final.target.nunique()} targets | {final.family.nunique()} families')
print(f'EXPECTED :  3011 records |   719 molecules |  47 targets | 18 families  (ChEMBL v36)')
print(f'\nwrote {OUT_CSV}')
final.head()

### Notes
* This reproduces the thesis's `curated_15desc.csv`. Small deviations from **3,011 / 719** almost
  always trace to the ChEMBL release used in Notebook 1 (the client fetches the latest ChEMBL); the
  thesis is pinned to **v36 (October 2025)**.
* `LE = 1.37 x pKi / HAC` is the standard ligand-efficiency form (binding free energy per heavy atom);
  `LLE = pKi - LogP`; `LELP = LogP / LE`.
* Descriptor MW here is RDKit `MolWt` on the salt-stripped parent, which is the descriptor reported in
  the thesis; the MW < 700 *filter* in Notebook 1 uses ChEMBL `full_mwt` (they agree except for a few
  salt forms near the 700 Da boundary).